# Train and validate the failure signals on Colab

This notebook is self contained. It trains the LoveDA baseline once, fits the novelty reference, validates the label free signals in domain on LoveDA, then runs the cross domain test on ISPRS Potsdam, all in one session with no dependency on a checkpoint saved earlier.

Switch the runtime to a GPU first. Use the menu Runtime then Change runtime type then pick a GPU such as T4. The full pass takes under an hour on a T4.

The result that matters comes from the last validation cell. If AUROC and the correlations stay clearly above chance on Potsdam, the label free signals hold up under geographic domain shift.

## 1. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "cuda available", torch.cuda.is_available())
!nvidia-smi -L

## 2. Get the code and the extras

Colab already ships a CUDA build of torch and torchvision, so those are not reinstalled.

In [ ]:
!git clone https://github.com/ShanmukhUpad/semantic-segmentation-feature.git
%cd semantic-segmentation-feature
!pip -q install seaborn tensorboard pyyaml tqdm scikit-learn gradio

## 3. Download LoveDA

The training tiles feed the novelty reference and the validation tiles feed the in domain baseline, so the dataset is needed even without training.

In [ ]:
!python scripts/download_loveda.py --root data/raw/LoveDA

## 4. Train the baseline

This trains DeepLabV3 ResNet50 on LoveDA with mixed precision at image size 384, which finishes 15 epochs in about 40 minutes on a T4. The best checkpoint by validation mIoU is written to results/loveda_deeplabv3/checkpoints/best.pth and used by every cell below.

In [ ]:
!python scripts/train.py --config configs/default.yaml --set train.epochs=15 dataloader.batch_size=8 dataloader.num_workers=2 train.amp=true dataset.image_size=[384,384]

## 5. Fit the novelty reference

This builds a picture of what LoveDA training tiles look like in backbone feature space. New scenes are later scored by how far they sit from this reference.

In [ ]:
!python scripts/fit_ood.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --num-images 300

## 6. Validate in domain on LoveDA

This measures how well each label free signal ranks wrong pixels above correct ones on the labeled validation split. AUROC of 0.5 is chance and 1.0 is perfect. It also writes the per image table that the cross domain cell uses as the novelty baseline.

In [ ]:
!python scripts/validate_signals.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --limit 300

import json
from IPython.display import Image as IPImage, display
base = "results/loveda_deeplabv3/validation/loveda_val"
print(json.dumps(json.load(open(base + "/metrics.json"))["pixel"], indent=2))
for name in ["risk_coverage.png", "failure_deciles.png", "failure_vs_error.png"]:
    display(IPImage(base + "/" + name))

## 7. Cross domain test on ISPRS Potsdam

This is the real proof. The model trained on Chinese imagery while Potsdam is German urban imagery at 0.05 m per pixel, so the geography, the sensor and the scene style all change at once. The cell pulls the ISPRS benchmark from a public Kaggle mirror, cuts the 38 orthophotos into 2304 pixel patches so the resized network input lands back at 0.3 m per pixel, then reruns the validation with the Potsdam class mapping and the LoveDA run above as the novelty baseline.

In [ ]:
import kagglehub
src = kagglehub.dataset_download("aletbm/urban-segmentation-isprs")
print("dataset at", src)

!python scripts/prepare_potsdam.py --images "{src}/Potsdam/Images" --labels "{src}/Potsdam/Labels" --output data/raw/PotsdamPatches

!python scripts/validate_signals.py --checkpoint results/loveda_deeplabv3/checkpoints/best.pth --set dataset.name=potsdam dataset.root=data/raw/PotsdamPatches --label-map potsdam_to_loveda --baseline-csv results/loveda_deeplabv3/validation/loveda_val/per_image.csv

import json
from IPython.display import Image as IPImage, display
base = "results/loveda_deeplabv3/validation/potsdam_val"
print(json.dumps(json.load(open(base + "/metrics.json"))["pixel"], indent=2))
for name in ["risk_coverage.png", "failure_deciles.png", "failure_vs_error.png", "novelty_hist.png"]:
    display(IPImage(base + "/" + name))

## 8. Save the run to Drive

Colab sessions are temporary. This mounts Drive and copies the whole run, so the checkpoint and every validation artifact survive the session ending. Run it once the results above look right.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r results/loveda_deeplabv3 "/content/drive/MyDrive/loveda_deeplabv3"
print("saved the run to Google Drive")